### Load gold partial + weather

In [6]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
INTERIM_DIR = REPO_ROOT / "data" / "interim"

gold = pd.read_parquet(PROCESSED_DIR / "gold_nbhd_day_partial.parquet")
weather = pd.read_parquet(INTERIM_DIR / "weather_obs_daily.parquet")

print("gold:", gold.shape)
print("weather:", weather.shape)

gold.head(2)

gold: (692514, 13)
weather: (4383, 10)


,AREA_ID,AREA_NAME,date,collisions,area_id,nbhd_id,area_name,ksi_collisions,ksi_fatal_collisions,ksi_serious_collisions,ksi_fatal_victims,ksi_victim_count,ksi_weighted_score
0,2502366,South Eglinton-Davisville,2014-01-01,0,2502366,174,South Eglinton-Davisville,0,0,0,0,0,0
1,2502366,South Eglinton-Davisville,2014-01-02,0,2502366,174,South Eglinton-Davisville,0,0,0,0,0,0


### Ensure date types match

In [7]:
gold["date"] = pd.to_datetime(gold["date"]).dt.date
weather["date"] = pd.to_datetime(weather["date"]).dt.date

### Merge

In [8]:
before = len(gold)
gold_w = gold.merge(weather, on="date", how="left")

assert len(gold_w) == before, "Row count changed — should not happen"

missing_weather = gold_w["tavg"].isna().sum()
print("Missing weather rows:", missing_weather)
if missing_weather > 0:
    # show some missing dates
    print(gold_w.loc[gold_w["tavg"].isna(), ["date"]].drop_duplicates().head(10))
    raise ValueError("Weather missing for some dates — fix weather extraction/date range.")

Missing weather rows: 0


### Save final gold

In [9]:
out_path = PROCESSED_DIR / "gold_nbhd_day_weather.parquet"
gold_w.to_parquet(out_path, index=False)
print("Saved:", out_path, "shape:", gold_w.shape)

Saved: C:\code\pyspark-playground\Covercheck-Toronto\data\processed\gold_nbhd_day_weather.parquet shape: (692514, 22)
